In [29]:
import os
import pandas as pd

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset

In [30]:
df = pd.read_csv("data/day.csv")
print(df.shape)
df.head()

(731, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [31]:
df.describe()

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
count,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000,731.000000
mean,366.000000,2.496580,0.500684,6.519836,0.028728,2.997264,0.683995,1.395349,0.495385,0.474354,0.627894,0.190486,848.176471,3656.172367,4504.348837
std,211.165812,1.110807,0.500342,3.451913,0.167155,2.004787,0.465233,0.544894,0.183051,0.162961,0.142429,0.077498,686.622488,1560.256377,1937.211452
min,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.059130,0.079070,0.000000,0.022392,2.000000,20.000000,22.000000
25%,183.500000,2.000000,0.000000,4.000000,0.000000,1.000000,0.000000,1.000000,0.337083,0.337842,0.520000,0.134950,315.500000,2497.000000,3152.000000
50%,366.000000,3.000000,1.000000,7.000000,0.000000,3.000000,1.000000,1.000000,0.498333,0.486733,0.626667,0.180975,713.000000,3662.000000,4548.000000
75%,548.500000,3.000000,1.000000,10.000000,0.000000,5.000000,1.000000,2.000000,0.655417,0.608602,0.730209,0.233214,1096.000000,4776.500000,5956.000000
max,731.000000,4.000000,1.000000,12.000000,1.000000,6.000000,1.000000,3.000000,0.861667,0.840896,0.972500,0.507463,3410.000000,6946.000000,8714.000000


In [32]:
# instant is just a row number and dteday is the date, so they are not useful for drift
# yr is kept for now because we use it to split the data in the next cell
df = df.drop(columns=["instant", "dteday"])

In [33]:
ref_2011 = df[df["yr"] == 0].drop(columns=["yr"])
cur_2012 = df[df["yr"] == 1].drop(columns=["yr"])

print("2011 rows:", len(ref_2011))
print("2012 rows:", len(cur_2012))

2011 rows: 365
2012 rows: 366


In [34]:
print("avg daily rentals 2011:", ref_2011["cnt"].mean())
print("avg daily rentals 2012:", cur_2012["cnt"].mean())

avg daily rentals 2011: 3405.7616438356163
avg daily rentals 2012: 5599.934426229508


In [35]:
schema = DataDefinition(
    numerical_columns=["temp", "atemp", "hum", "windspeed", "casual", "registered", "cnt"],
    categorical_columns=["season", "mnth", "holiday", "weekday", "workingday", "weathersit"],
)

In [36]:
ref_data = Dataset.from_pandas(ref_2011, data_definition=schema)
cur_data = Dataset.from_pandas(cur_2012, data_definition=schema)

In [ ]:
report = Report([
    DataDriftPreset(),
    DataSummaryPreset()
])

# first argument is current data, second is reference
year_eval = report.run(cur_data, ref_data)
year_eval

In [38]:
os.makedirs("reports", exist_ok=True)
year_eval.save_html("reports/drift_2011_vs_2012.html")

In [ ]:
half_a = df.drop(columns=["yr"]).sample(frac=0.5, random_state=42)
half_b = df.drop(columns=["yr"]).drop(half_a.index)

a_data = Dataset.from_pandas(half_a, data_definition=schema)
b_data = Dataset.from_pandas(half_b, data_definition=schema)

random_eval = Report([DataDriftPreset()]).run(b_data, a_data)
random_eval

In [40]:
random_eval.save_html("reports/drift_random_split.html")

In [ ]:
fake_current = ref_2011.copy()
fake_current["hum"] = fake_current["hum"] * 1.3

fake_data = Dataset.from_pandas(fake_current, data_definition=schema)

sensor_eval = Report([DataDriftPreset()]).run(fake_data, ref_data)
sensor_eval

In [42]:
sensor_eval.save_html("reports/drift_fake_humidity.html")